In [ ]:
# Fig1A picture drawing
"""
Description:
1. Load OEWS 2023 all-sector data
2. Extract 4-digit NAICS codes
3. Filter rows by the stranded industry definition
4. Skip filtering OCC_CODE == 00-0000 (no aggregate occupation row in the file)
5. Aggregate stranded-industry employment by state
6. Plot a US state-level choropleth map
7. Export CSV, HTML, and PDF
8. Increase title / colorbar / tick / hover font sizes
"""

import pandas as pd
import plotly.express as px
from pathlib import Path

# =========================
# 1. File paths
# =========================
file_path = Path("Your path/Stranded_Occupations_Replication/Data/raw_new/oes_wage/oes_research_2023_allsectors.xlsx")

out_csv = Path("Your path/Stranded_Occupations_Replication/Data/temp/stranded_state_employment_2023.csv")
out_html = Path("Your path/Stranded_Occupations_Replication/Data/temp/stranded_state_employment_2023_map.html")
out_pdf = Path("Your path/Stranded_Occupations_Replication/Data/temp/stranded_state_employment_2023_map.pdf")

# =========================
# 2. Stranded industry definition (4-digit NAICS)
# =========================
stranded_naics4 = {
    "2111", "2121", "2131", "2211", "2212", "2371",
    "3241", "3251", "3331", "3363", "4235", "4247",
    "4571", "4572", "4861", "4862", "4869"
}

# =========================
# 3. Load data
# =========================
df = pd.read_excel(file_path, dtype=str)
df.columns = [c.strip() for c in df.columns]

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

# =========================
# 4. Check required columns
# =========================
required_cols = ["AREA_TITLE", "NAICS", "TOT_EMP"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# =========================
# 5. Clean TOT_EMP
# =========================
df["TOT_EMP"] = (
    df["TOT_EMP"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("*", "", regex=False)
    .str.replace("#", "", regex=False)
    .str.strip()
)
df["TOT_EMP"] = pd.to_numeric(df["TOT_EMP"], errors="coerce")

# =========================
# 6. Extract 4-digit NAICS
# =========================
df["NAICS"] = df["NAICS"].astype(str).str.extract(r"(\d+)", expand=False)
df["NAICS4"] = df["NAICS"].str[:4]

# =========================
# 7. Keep only stranded industries
# =========================
df = df[df["NAICS4"].isin(stranded_naics4)].copy()

print("Rows after filtering for stranded industries:", len(df))

# =========================
# 8. Inspect O_GROUP if present
#    All rows are detail-level here, so no extra filtering
# =========================
if "O_GROUP" in df.columns:
    print("Unique values of O_GROUP:", df["O_GROUP"].dropna().unique())

# =========================
# 9. Clean state names
# =========================
df["AREA_TITLE"] = df["AREA_TITLE"].str.strip()

exclude_areas = {
    "United States"
    # To exclude DC, also add "District of Columbia"
}
df = df[~df["AREA_TITLE"].isin(exclude_areas)].copy()

# =========================
# 10. Map state names to two-letter abbreviations
# =========================
state_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY"
}

# =========================
# 11. Identify which AREA_TITLE values map to valid states
# =========================
df["state_abbrev"] = df["AREA_TITLE"].map(state_abbrev)

unmatched_area = sorted(df.loc[df["state_abbrev"].isna(), "AREA_TITLE"].dropna().unique().tolist())
if unmatched_area:
    print("The following AREA_TITLE values are not standard state names and have been excluded (first 50):")
    print(unmatched_area[:50])

# Keep only records matching a valid state abbreviation
df = df[df["state_abbrev"].notna()].copy()

print("State-level records available for mapping:", len(df))

# =========================
# 12. Aggregate employment by state
# =========================
state_emp = (
    df.groupby(["AREA_TITLE", "state_abbrev"], as_index=False)["TOT_EMP"]
      .sum(min_count=1)
      .rename(columns={
          "AREA_TITLE": "state_name",
          "TOT_EMP": "stranded_emp"
      })
      .sort_values("stranded_emp", ascending=False)
)

# Save the state-level summary table
state_emp.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"State-level summary table saved: {out_csv}")

# =========================
# 13. Draw the choropleth map
# =========================
fig = px.choropleth(
    state_emp,
    locations="state_abbrev",
    locationmode="USA-states",
    color="stranded_emp",
    scope="usa",
    hover_name="state_name",
    hover_data={
        "state_abbrev": False,
        "stranded_emp": ":,.0f"
    },
    color_continuous_scale="OrRd",
    title="Employment in Stranded Industries by State, 2023"
)

# =========================
# 14. Increase font sizes and adjust layout
# =========================
fig.update_layout(
    width=1400,
    height=900,
    title=dict(
        text="Employment in Stranded Industries by State, 2023",
        x=0.5,
        xanchor="center",
        font=dict(size=28)
    ),
    font=dict(size=18),
    margin=dict(l=30, r=30, t=90, b=30),
    coloraxis_colorbar=dict(
        title=dict(
            text="Number of laborforce",
            font=dict(size=22)
        ),
        tickfont=dict(size=18),
        len=0.8,
        thickness=28
    )
)

# Increase hover label font size
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>Employment: %{z:,.0f}<extra></extra>",
    hoverlabel=dict(font_size=18)
)

# =========================
# 15. Export HTML
# =========================
fig.write_html(str(out_html))
print(f"Interactive map saved: {out_html}")

# =========================
# 16. Export PDF
#    Requires kaleido:
#    pip install -U kaleido
# =========================
try:
    fig.write_image(str(out_pdf), format="pdf", width=1400, height=900, scale=2)
    print(f"PDF map saved: {out_pdf}")
except Exception as e:
    print("PDF export failed, usually because kaleido is not installed.")
    print("Please run: pip install -U kaleido")
    print("Error message: ", e)

# =========================
# 17. Print top-10 states
# =========================
print("\nTop 10 states by stranded-industry employment:")
print(state_emp.head(10))

In [ ]:
# Fig1B picture drawing
"""
Description:
1. Load naics_nem_new.dta
2. Keep only the four occupation types with valid type_skill_all:
   - Non-Routine Cognitive
   - Routine Manual
   - Non-Routine Manual
   - Routine Cognitive
3. Compute separately:
   - strand_ind == 1 (stranded industries)
   - strand_ind == 0 (non-stranded industries)
   employment share of the four occupation types
4. Sort segments within each bar by share (descending) and stack bottom-up
5. Widen the gap between the two bars to avoid x-axis label overlap
6. Add color-coded legend entries: NRC, RC, NRM, RM
7. Label each stacked segment with its percentage in the center
8. Export SVG and PNG
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

# =========================
# 1. File paths
# =========================
file_path = Path("Your path/Stranded_Occupations_Replication/Data/temp/naics_nem_new.dta")

out_svg = Path("Your path/Stranded_Occupations_Replication/Data/temp/Fig1B.svg")
out_png = Path("Your path/Stranded_Occupations_Replication/Data/temp/Fig1B.png")

# =========================
# 2. Load data
# =========================
df = pd.read_stata(file_path)

# Strip whitespace from column names
df.columns = [c.strip() for c in df.columns]

# Strip whitespace from string columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# =========================
# 3. Keep only the four occupation types
# =========================
skill_keep = [
    "Non-Routine Cognitive",
    "Routine Manual",
    "Non-Routine Manual",
    "Routine Cognitive"
]

# Abbreviation label map
abbr_map = {
    "Non-Routine Cognitive": "NRC",
    "Routine Cognitive": "RC",
    "Non-Routine Manual": "NRM",
    "Routine Manual": "RM"
}

df["type_skill_all"] = df["type_skill_all"].astype(str).str.strip()
df["Employment"] = pd.to_numeric(df["Employment"], errors="coerce")
df["strand_ind"] = pd.to_numeric(df["strand_ind"], errors="coerce")

df = df[
    df["type_skill_all"].isin(skill_keep) &
    df["Employment"].notna() &
    df["strand_ind"].notna()
].copy()

# =========================
# 4. Aggregate employment
# =========================
summary = (
    df.groupby(["strand_ind", "type_skill_all"], as_index=False)["Employment"]
      .sum()
)

summary["share"] = (
    summary.groupby("strand_ind")["Employment"]
    .transform(lambda x: x / x.sum() * 100)
)

# =========================
# 5. Separate BS (stranded) and RS (non-stranded)
# =========================
bs = summary[summary["strand_ind"] == 1][["type_skill_all", "Employment", "share"]].copy()
rs = summary[summary["strand_ind"] == 0][["type_skill_all", "Employment", "share"]].copy()

bs = (
    bs.set_index("type_skill_all")
      .reindex(skill_keep)
      .fillna(0)
      .reset_index()
)
rs = (
    rs.set_index("type_skill_all")
      .reindex(skill_keep)
      .fillna(0)
      .reset_index()
)

# Sort each group by share descending
bs = bs.sort_values("share", ascending=False).reset_index(drop=True)
rs = rs.sort_values("share", ascending=False).reset_index(drop=True)

# =========================
# 6. Color settings
# =========================
color_map = {
    "Non-Routine Cognitive": "#8EA7D3",  # blue
    "Routine Cognitive": "#A8CF8B",      # green
    "Non-Routine Manual": "#E9D98B",     # yellow
    "Routine Manual": "#D6CFC7"          # beige-gray
}

# =========================
# 7. Draw stacked bar chart with percentage labels
# =========================
fig, ax = plt.subplots(figsize=(6.2, 7.2), dpi=300)

x_positions = [0, 2.2]
x_labels = ["Stranded", "Non-stranded"]
bar_width = 0.72

# Skip labeling segments below this threshold to avoid crowding
label_threshold = 4

# ---- Draw BS (stranded) bar ----
bottom = 0
for _, row in bs.iterrows():
    skill = row["type_skill_all"]
    share = row["share"]

    ax.bar(
        x_positions[0],
        share,
        bottom=bottom,
        width=bar_width,
        color=color_map.get(skill, "#CCCCCC"),
        edgecolor="white",
        linewidth=1.6
    )

    # Place the percentage label at the center of the current segment
    if share >= label_threshold:
        y_center = bottom + share / 2
        ax.text(
            x_positions[0],
            y_center,
            f"{share:.1f}%",
            ha="center",
            va="center",
            fontsize=13,
            color="black"
        )

    bottom += share

# ---- Draw RS (non-stranded) bar ----
bottom = 0
for _, row in rs.iterrows():
    skill = row["type_skill_all"]
    share = row["share"]

    ax.bar(
        x_positions[1],
        share,
        bottom=bottom,
        width=bar_width,
        color=color_map.get(skill, "#CCCCCC"),
        edgecolor="white",
        linewidth=1.6
    )

    # Place the percentage label at the center of the current segment
    if share >= label_threshold:
        y_center = bottom + share / 2
        ax.text(
            x_positions[1],
            y_center,
            f"{share:.1f}%",
            ha="center",
            va="center",
            fontsize=13,
            color="black"
        )

    bottom += share

# =========================
# 8. Legend
# =========================
legend_order = [
    "Non-Routine Cognitive",
    "Routine Cognitive",
    "Non-Routine Manual",
    "Routine Manual"
]

legend_handles = [
    Patch(facecolor=color_map[s], edgecolor="none", label=abbr_map[s])
    for s in legend_order
]

ax.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    frameon=False,
    fontsize=15,
    handlelength=1.4,
    labelspacing=0.8,
    borderaxespad=0.0
)

# =========================
# 9. Beautify the figure
# =========================
ax.set_ylim(0, 100)
ax.set_xlim(-0.8, 3.0)

ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels, fontsize=15)

ax.set_yticks([0, 25, 50, 75, 100])
ax.set_yticklabels([0, 25, 50, 75, 100], fontsize=18)

ax.set_ylabel("Employment share (%)", fontsize=20)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.6)
ax.spines["bottom"].set_linewidth(1.6)

ax.tick_params(axis="both", width=1.6, length=6, pad=10)

ax.set_facecolor("white")
fig.patch.set_facecolor("white")

plt.tight_layout()

# =========================
# 10. Export the figure
# =========================
plt.savefig(out_svg, format="svg", bbox_inches="tight")
plt.savefig(out_png, format="png", dpi=400, bbox_inches="tight")
plt.show()

# =========================
# 11. Print result tables
# =========================
print("\nBS (strand_ind = 1) occupation type shares:")
print(bs)

print("\nRS (strand_ind = 0) occupation type shares:")
print(rs)

print("\nFiles exported:")
print(out_svg)
print(out_png)